<a href="https://colab.research.google.com/github/mn660875/NetSol-Challenges/blob/main/Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-08 06:19:41--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  6.13MB/s    in 0.2s    

2026-09-08 06:19:41 (6.13 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
import re
from collections import defaultdict, Counter


# ---------- STEP 1: Load and clean text (aapka pehla code) ----------

def load_data(filepath):
    """Load text file and return as string"""
    with open(filepath, 'r', encoding='utf-8') as file:
        text = file.read()
    return text


def preprocess(text):
    """
    Preprocess text by:
    1. Converting to lowercase
    2. Removing special characters
    3. Splitting into tokens (words)
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9'\s]", "", text)
    tokens = text.split()
    return tokens



In [3]:
def build_ngram_model(tokens, n=2):
    """
    Build n-gram model from tokens.
    Example: For n=2 (bigram), returns {"i": {"am": 5, "have": 3}}

    Idea: slide a window of size n over the tokens.
    The first (n-1) words = "context", the last word = "next word".
    We count how many times each "next word" follows each context.
    """
    model = defaultdict(Counter)

    # zip(*[tokens[i:] for i in range(n)]) creates sliding windows of size n
    # Example n=2, tokens=[i, am, happy]:
    #   tokens[0:] = [i, am, happy]
    #   tokens[1:] = [am, happy]
    #   zip gives: (i, am), (am, happy)
    for window in zip(*[tokens[i:] for i in range(n)]):
        context = window[:-1]      # everything except last word
        next_word = window[-1]     # last word

        # for bigrams (n=2), context is a 1-word tuple -> unwrap it to a plain string
        if n == 2:
            context = context[0]

        model[context][next_word] += 1

    return model



In [14]:
def predict_next_word(model, context, k=3):
    """
    Predict next words given context.
    Returns: list of (word, probability) tuples, sorted by probability descending.
    """
    if context not in model:
        return []  # we've never seen this context in training data

    counts = model[context]                 # e.g. {"happy": 2, "sad": 1}
    total = sum(counts.values())            # total times this context appeared

    # convert counts -> probabilities
    probabilities = [(word, count / total) for word, count in counts.items()]

    # sort by probability, highest first, and keep top k
    probabilities.sort(key=lambda pair: pair[1], reverse=True)

    return probabilities[:k]


In [16]:
if __name__ == "__main__":
    text = load_data("input.txt")
    tokens = preprocess(text)
    print("First 50 tokens:", tokens[:50])

    model = build_ngram_model(tokens, n=2)

    # Try predicting next word for a sample context word.
    # Change "the" to any word you expect to be in your input.txt
    sample_context = "before"
    predictions = predict_next_word(model, sample_context, k=3)

    print(f"\nTop predictions after '{sample_context}':")
    for word, prob in predictions:
        print(f"  {word}  (probability: {prob:.2f})")



First 50 tokens: ['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'speak', 'speak', 'first', 'citizen', 'you', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish', 'all', 'resolved', 'resolved', 'first', 'citizen', 'first', 'you', 'know', 'caius', 'marcius', 'is', 'chief', 'enemy', 'to', 'the', 'people', 'all', 'we', "know't", 'we', "know't", 'first', 'citizen', 'let', 'us']

Top predictions after 'before':
  i  (probability: 0.12)
  the  (probability: 0.11)
  him  (probability: 0.07)
